# Training and Experiments

Here we train the models and perform sigma tuning on the RNN-POA model.

In [ ]:
import sys
import os
import json
sys.path.append('..')

import torch
import numpy as np
from datasets import load_dataset
from torch.utils.data import DataLoader

from utils.data_utils import *
from utils.model_utils import RNNAVG, RNNATT, RNNPOA
from utils.train_utils import train_model, evaluate, compute_metrics

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

os.makedirs('../results', exist_ok=True)

### Prepare Data

In [ ]:
wiki_qa = load_dataset('wiki_qa')
word2idx, idx2word = build_vocab(wiki_qa['train'], [])
embedding_matrix = build_embedding_matrix(word2idx, load_glove())

BATCH_SIZE = 64
train_ds = QADataset(wiki_qa['train'], word2idx)
dev_ds   = QADataset(wiki_qa['validation'], word2idx)
test_ds  = QADataset(wiki_qa['test'], word2idx)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
dev_loader   = DataLoader(dev_ds,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

### Train Baselines: RNN-AVG and RNN-ATT

In [ ]:
HIDDEN_DIM = 50
N_EPOCHS = 10 # Set to 10 for quick testing, increase for real results

model_avg = RNNAVG(embedding_matrix, hidden_dim=HIDDEN_DIM).to(device)
model_att = RNNATT(embedding_matrix, hidden_dim=HIDDEN_DIM).to(device)

print('Training RNN-AVG...')
hist_avg = train_model(model_avg, train_loader, dev_loader, n_epochs=N_EPOCHS, save_name='../results/best_avg.pt')

print('\nTraining RNN-ATT...')
hist_att = train_model(model_att, train_loader, dev_loader, n_epochs=N_EPOCHS, save_name='../results/best_att.pt')

### Sigma Tuning for RNN-POA
We tune the $\sigma$ parameter from [5, 15, 25, 35, 45, 55] as requested.

In [ ]:
sigmas = [5, 15, 25, 35, 45, 55]
poa_results = {}
best_sigma = None
best_map = 0.0

for sig in sigmas:
    print(f'\n--- Training RNN-POA with sigma={sig} ---')
    model = RNNPOA(embedding_matrix, hidden_dim=HIDDEN_DIM, sigma_scope=sig).to(device)
    hist = train_model(model, train_loader, dev_loader, n_epochs=N_EPOCHS, save_name=f'../results/poa_sig_{sig}.pt')
    
    # Evaluate on test set
    _, test_map, test_mrr = evaluate(model, test_loader)
    poa_results[sig] = {'MAP': test_map, 'MRR': test_mrr}
    print(f'Sigma={sig} Test MAP: {test_map:.4f}, MRR: {test_mrr:.4f}')
    
    if test_map > best_map:
        best_map = test_map
        best_sigma = sig

print(f'\nBest Sigma: {best_sigma} with MAP: {best_map:.4f}')

### Evaluate Baselines on Test Set

In [ ]:
_, avg_map, avg_mrr = evaluate(model_avg, test_loader)
_, att_map, att_mrr = evaluate(model_att, test_loader)

results = {
    'AVG': {'MAP': avg_map, 'MRR': avg_mrr},
    'ATT': {'MAP': att_map, 'MRR': att_mrr},
    'POA_Tuning': poa_results,
    'POA_Best': poa_results[best_sigma]
}

with open('../results/experiment_results.json', 'w') as f:
    json.dump(results, f)

print('Experiments completed and results saved.')